In [2]:
#autoreload
%load_ext autoreload
%autoreload 2
import torch
import matplotlib.pyplot as plt
import numpy as np
import featureman.gen_data as man
import featureman.utils as utils
from sklearn.cluster import SpectralClustering
import pickle
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
model_dict = torch.load("modular_arithmetic_model.pth", map_location=device)
model = man.OneLayerTransformer(p=113, d_model=128, nheads=4).to(device)
model.load_state_dict(model_dict)

<All keys matched successfully>

In [4]:
torch.manual_seed(1337)
# generate combination of all inputs a and b range (113)
a_values = np.arange(113)
b_values = np.arange(113)
# generate inputs for the model
inputs = np.array([[a_i, 113, b_i, 114] for a_i in a_values for b_i in b_values])
inputs = torch.tensor(inputs).to(device)  # Add batch dimension

logits, activations = model(inputs, return_activations=True)
activation_final = activations[:, -1, :].detach()
batched_acts = activation_final.unsqueeze(0).repeat(5, 1, 1).to(device)

In [7]:
# Import the module
import featureman.reducibility as irr
import pandas as pd
from datetime import datetime
from sklearn.decomposition import PCA

# Your updated loop
all_summaries = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f"irreducibility_results_{timestamp}"

reconstructions = batched_acts[3].detach().cpu().numpy()

# Run silent analysis
summary = irr.analyze_cluster_irreducibility_silent(
    reconstructions,
    reconstructions,
    cluster_idx=1, 
    save_plots=True, 
    save_dir=save_dir
)

all_summaries.append(summary)

# Concise output
irreducible_flag = "✅" if summary['is_irreducible'] else "❌"

# Save final summary
print(f"\n{'='*60}")
df, csv_path = irr.save_analysis_summary(all_summaries, save_dir)
print(f"🎯 Analysis complete! Check {save_dir}/ for all plots and {csv_path} for summary.")


📊 Summary saved to: irreducibility_results_20250823_230219/irreducibility_summary_20250823_230222.csv
🏆 Top 5 irreducible clusters:
   cluster_idx  irreducibility_score  mean_separability  mean_mixture  \
0            1              0.295036           0.450793      0.345517   

   is_irreducible  
0            True  
🎯 Analysis complete! Check irreducibility_results_20250823_230219/ for all plots and irreducibility_results_20250823_230219/irreducibility_summary_20250823_230222.csv for summary.
